In [1]:
#
# Copyright (c) 2024, NVIDIA CORPORATION.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#

<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/tensorrt_torchtrt_efficientnet/nvidia_logo.png" width="90px">

# Distributed Hyperparameter Tuning: Optuna + Spark RDD


This demo demonstrates distributed hyperparameter tuning for XGBoost using Spark RDDs and Spark barrier stages to achieve **deterministic tuning results**.  
We implement best practices to precompute data and maximize computations on the GPU.  



Reference: https://forecastegy.com/posts/xgboost-hyperparameter-tuning-with-optuna/

#### Note:
Before running, please make sure you've followed the relevant [setup instructions](../README.md) for your environment (standalone or databricks).


In [2]:
from typing import List, Dict, Any
import os
import requests
import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from pyspark.sql import SparkSession
from pyspark import TaskContext, SparkConf

### Download the dataset

We'll use the [red wine quality dataset](https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv) to regress wine quality based on features such as acidity, sugar content, etc.  

**Note**: This example uses a small dataset for demonstration purposes. The performance advantages of distributed training are best realized with large datasets and computational workloads.

In [3]:
def download_file(filepath, url):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"File downloaded and saved to {filepath}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

**For Databricks**: we must download the dataset to DBFS so that all workers can access it.

In [4]:
on_databricks = os.environ.get("DATABRICKS_RUNTIME_VERSION", False)

In [5]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

if on_databricks:
    dbutils.fs.mkdirs("/FileStore/optuna-data")
    filepath = "/dbfs/FileStore/optuna-data/winequality-red.csv"
    download_file(filepath, url)
else:
    cwd = os.getcwd()
    os.mkdir(os.path.join(cwd, "data")) if not os.path.exists(os.path.join(cwd, "data")) else None
    filepath = os.path.join(cwd, "data", "winequality-red.csv")
    download_file(filepath, url)

File downloaded and saved to /home/rishic/Code/myforks/spark-rapids-examples/examples/ML+DL-Examples/Optuna-Spark/optuna-examples/data/winequality-red.csv


## Distributed Optuna on Spark 

### PySpark

For standalone users, we need to create the Spark session. For Databricks users, the Spark session will be preconfigured.

In [6]:
def initialize_spark():
    import socket
    hostname = socket.gethostname()
    conda_env = os.environ.get("CONDA_PREFIX")

    conf = SparkConf()
    conf.setMaster(f"spark://{hostname}:7077")  # Assuming master is on host and default port. 
    conf.set("spark.task.maxFailures", "1")
    conf.set("spark.task.resource.gpu.amount", f"{1/4}")  # Setting to 1/4 for single-node demo. In practice, set to 1. 
    conf.set("spark.executor.resource.gpu.amount", "1")
    conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
    conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
    
    spark = SparkSession.builder.appName("optuna-deterministic-xgboost").config(conf=conf).getOrCreate()
    return spark

if 'spark' not in globals():
    spark = initialize_spark()

24/12/12 18:54:22 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
24/12/12 18:54:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/12 18:54:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Optuna Task

This implementation demonstrates **Worker I/O**. 

This means that each worker will read the full dataset from the filepath rather than passing the data in a dataframe.  
In practice, this requires the dataset to be written to a distributed file system accessible to all workers prior to tuning. 

For the alternative implementation using **Spark I/O**, see the [Spark Dataframe notebook](optuna-dataframe.ipynb).

#### Determinism

To achieve determinism, we take the following steps:  
- Each of the n workers creates a **local Optuna study**. 
- At the start of each iteration, each worker will initialize n new trials in their local study, but only execute the trial associated with their worker ID. 
- At the end of each iteration, the workers perform a barrier.allgather() to synchronize and get trial results from all workers.
- The workers update the n trials with these results in a deterministic order (using Optuna's [ask-and-tell interface](https://optuna.readthedocs.io/en/stable/tutorial/20_recipes/009_ask_and_tell.html)).
- Finally, one worker will save the study to MySQL for persistent storage.

In [7]:
def task(xgb_params: Dict[str, Any],
         optuna_params: Dict[str, optuna.distributions.BaseDistribution],
         trials_per_task: List[int],
         driver_ip: str,
         study_name: str,
         seed: int,
         filepath: str):
    import json
    import cudf
    from cuml.metrics.regression import mean_squared_error
    from cuml.model_selection import train_test_split
    from pyspark import BarrierTaskContext
    from pyspark import TaskContext

    task_context = TaskContext.get()
    barrier_taskcontext = BarrierTaskContext.get()
    part_id = barrier_taskcontext.partitionId()

    assert "gpu" in task_context.resources(), "GPU resource not found."

    num_trials = trials_per_task[part_id]
    num_workers = len(trials_per_task)
    base_size = sum(trials_per_task) // num_workers
    workers_with_extra_trials = [i for i, trials in enumerate(trials_per_task) if trials > base_size]

    print(f"DETERMINISTIC: Running {num_trials} trials on task {part_id} on dataset {filepath}")

    data = cudf.read_csv(filepath, delimiter=";")
    X = data.iloc[:, :-1].values
    y = data["quality"].values
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    tuning_max_bin = "max_bin" in optuna_params
    if not tuning_max_bin:
        max_bin = xgb_params.get("max_bin", 256)
        Xy_train_qdm = xgb.QuantileDMatrix(X_train, y_train, max_bin=max_bin)

    # Create local study for each worker
    try:
        optuna.delete_study(
            study_name=f"optuna-spark-xgboost-worker-{part_id}"
        )
    except:
        pass
    
    sampler = TPESampler(seed=seed)
    worker_study = optuna.create_study(
        study_name=f"optuna-spark-xgboost-worker-{part_id}",
        sampler=sampler
    )

    ### Objective ###
    for trial_idx in range(1, max(trials_per_task)+1):
        worker_optuna_params = {}
        trial_numbers = []
        for i in range(num_workers):
            if trial_idx > base_size and i not in workers_with_extra_trials:
                # For uneven trial splits, skip this worker if it doesn't have an extra trial
                trial_numbers.append(-1)
                continue
            trial = worker_study.ask(optuna_params)
            trial_numbers.append(trial.number)
            if i == part_id:
                worker_optuna_params.update(trial.params)

        if trial_idx <= num_trials:
            xgb_params.update(worker_optuna_params)
            
            if tuning_max_bin:
                # If tuning the max_bin param, we must recompute the QDM every trial, since the quantiles change.
                if "n_estimators" not in xgb_params:
                    xgb_params["n_estimators"] = 100

                model = xgb.XGBRegressor(**xgb_params)
                model.fit(X_train, y_train)
                booster = model.get_booster()
            else:
                num_boost_round = xgb_params.get("n_estimators", 100)
                booster = xgb.train(params=xgb_params, dtrain=Xy_train_qdm, num_boost_round=num_boost_round)

            predictions = booster.inplace_predict(X_val)
            rmse = mean_squared_error(y_val, predictions, squared=False).get()
        else:
            rmse = -1

        # Synchronize and share results with allgather
        messages = barrier_taskcontext.allGather(message=json.dumps((part_id, float(rmse))))
        worker_results = [json.loads(message) for message in messages]

        # Update the study with the results from all workers in order
        for worker_id, worker_rmse in worker_results:
            if worker_rmse != -1:
                assert trial_numbers[worker_id] != -1  # safety check
                worker_study.tell(trial_numbers[worker_id], worker_rmse)

    # Have one worker save the study to MySQL
    if part_id == 0:
        global_study = optuna.load_study(
            study_name=study_name,
            storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna"
        )
        global_study.add_trials(worker_study.trials)

    yield (worker_study.best_params, worker_study.best_value, sampler)

## Setup and run the Optuna study

Get the driver IP for the MySQL database.  
- For standalone users, make sure you've followed the [database setup instructions](../README.md#setup-database-for-optuna). The database should be on 'localhost'. 
- For databricks users, the database should already be setup on the driver node by the init script.

In [8]:
if on_databricks:
    driver_ip = spark.conf.get("spark.driver.host")
else:
    driver_ip = "localhost"

print(f"MySQL database is hosted on {driver_ip}")

MySQL database is hosted on localhost


### Run 1

Create a new study, referencing the MySQL database as the storage backend.

In [9]:
run = 1
study_name = f"optuna-xgboost-deterministic-{run}"

try:
    optuna.delete_study(
        study_name=study_name, 
        storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna",
    )
except:
    pass

optuna.create_study(
    study_name=study_name,
    storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna"
)

[I 2024-12-12 18:54:23,900] A new study created in RDB with name: optuna-xgboost-deterministic-1


Define the number of tasks, number of trials, and trials per task. 

**NOTE**: for standalone users running on a single worker, the 4 tasks will all be assigned to the same worker and will time-share the GPU for demonstration. In practice, you should set `spark.task.resource.gpu.amount=1` and set num_tasks to the number of workers in the cluster so that each task gets full access to the GPU.

In [10]:
def partition_trials(total_trials: int, total_tasks: int) -> List[int]:
    base_size = total_trials // total_tasks
    extra = total_trials % total_tasks
    partitions = [base_size] * total_tasks
    for i in range(extra):
        partitions[i] += 1
    
    return partitions

In [11]:
num_tasks = 4
num_trials = 100
trials_per_task = partition_trials(num_trials, num_tasks)
print(f"Trials per task: {trials_per_task}")

Trials per task: [25, 25, 25, 25]


#### Define params
Define the XGBoost model params and the hyperparams for Optuna to tune. 

In [12]:
seed = 42  # Use this for XGBoost, Optuna sampler, and data splitting

In [13]:
xgb_params = {
    "objective": "reg:squarederror",
    "verbosity": 0,
    "tree_method": "gpu_hist",
    "device": f"cuda",
    "seed": seed,
}

In [ ]:
optuna_params = {
    "n_estimators": optuna.distributions.IntDistribution(100, 500),
    "learning_rate": optuna.distributions.FloatDistribution(1e-3, 0.1, log=True),
    "max_depth": optuna.distributions.IntDistribution(1, 10),
    "subsample": optuna.distributions.FloatDistribution(0.05, 1.0),
    "colsample_bytree": optuna.distributions.FloatDistribution(0.05, 1.0),
    "min_child_weight": optuna.distributions.IntDistribution(1, 20),
}

### Run the study
Map the Optuna task onto the RDD and collect the results (it might take a few minutes).

In [15]:
print(f"Running deterministic run {run}.")
dummy_rdd = spark.sparkContext.parallelize(range(num_tasks), numSlices=num_tasks)
results = dummy_rdd.barrier().mapPartitions(lambda _: 
                                            task(xgb_params=xgb_params,
                                                 optuna_params=optuna_params,
                                                 trials_per_task=trials_per_task,
                                                 driver_ip=driver_ip,
                                                 study_name=study_name,
                                                 seed=seed,
                                                 filepath=filepath)).collect()

Running deterministic run 1.


24/12/12 18:54:24 WARN DAGScheduler: Barrier stage in job 0 requires 4 slots, but only 0 are available. Will retry up to 40 more times


In [16]:
best_params_1, best_value_1, sampler_1 = results[0]
print(f"Best Params: {best_params_1}")
print(f"Best RMSE: {best_value_1}")
print(f"Sampler: {sampler_1}")

Best Params: {'n_estimators': 224, 'learning_rate': 0.07598392201206536, 'max_depth': 7, 'subsample': 0.9757365874547643, 'colsample_bytree': 0.7760550691250085, 'min_child_weight': 3}
Best RMSE: 0.5340637746858722
Sampler: TPESampler


We can reload the study for future use like so:

In [17]:
study = optuna.load_study(
    study_name="optuna-xgboost-deterministic-1",
    storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna",
    sampler=sampler_1
)

We can also export the full trial history in a pandas dataframe:

In [18]:
all_trials_1 = study.trials_dataframe()

### Run 2

Create a new study for the second run, reuse the same params. 

In [19]:
run = 2
study_name = f"optuna-xgboost-deterministic-{run}"

try:
    optuna.delete_study(
        study_name=study_name, 
        storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna",
    )
except:
    pass

optuna.create_study(
    study_name=study_name,
    storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna",
)

[I 2024-12-12 18:56:25,579] A new study created in RDB with name: optuna-xgboost-deterministic-2


In [20]:
print(f"Running deterministic run {run}.")
dummy_rdd = spark.sparkContext.parallelize(range(num_tasks), numSlices=num_tasks)
results = dummy_rdd.barrier().mapPartitions(lambda _: 
                                            task(xgb_params=xgb_params,
                                                 optuna_params=optuna_params,
                                                 trials_per_task=trials_per_task,
                                                 driver_ip=driver_ip,
                                                 study_name=study_name,
                                                 seed=seed,
                                                 filepath=filepath)).collect()

Running deterministic run 2.


In [21]:
best_params_2, best_value_2, sampler_2 = results[0]
print(f"Best Params: {best_params_2}")
print(f"Best RMSE: {best_value_2}")
print(f"Sampler: {sampler_2}")

Best Params: {'n_estimators': 224, 'learning_rate': 0.07598392201206536, 'max_depth': 7, 'subsample': 0.9757365874547643, 'colsample_bytree': 0.7760550691250085, 'min_child_weight': 3}
Best RMSE: 0.5340637746858722
Sampler: TPESampler


Like before, we can reload the study and get the trial history:

In [22]:
study = optuna.load_study(
    study_name="optuna-xgboost-deterministic-2",
    storage=f"mysql://optuna_user:optuna_password@{driver_ip}/optuna",
    sampler=sampler_2,
)

all_trials_2 = study.trials_dataframe()

### Test Correctness

A quick check shows that the study results from the two runs are the same.  
We can further ensure that the full study histories are identical (besides timestamps).

In [24]:
assert best_params_2 == best_params_1
assert best_value_2 == best_value_1

In [25]:
all_trials_1.drop(['datetime_start', 'datetime_complete', 'duration'], axis=1, inplace=True)
all_trials_2.drop(['datetime_start', 'datetime_complete', 'duration'], axis=1, inplace=True)

assert all_trials_1.equals(all_trials_2)
print("All trials matched.")

All trials matched.
